# 06 - 性能与事务（Performance & Transactions）
> Harvard CS50's Intro to Databases with SQL  |  课程时间：7:31:09 – 9:01:45

这章讲数据库的两大「生产级」能力：**索引加速查询**和**事务保证数据正确**。索引让 100 万行的查询从秒级变毫秒；事务让你的转账不会扣了钱没加上。

## 学习路线

| # | 内容 |
|---|------|
| 6.1 | Index — 索引，查询加速器 |
| 6.2 | EXPLAIN QUERY PLAN — 看数据库怎么执行查询 |
| 6.3 | VACUUM — 空间回收 |
| 6.4 | Transaction — 事务，BEGIN/COMMIT/ROLLBACK |
| 6.5 | ACID — 原子性/一致性/隔离性/持久性 |

> ⚠️ 先运行下面的 cell 创建练习环境！

In [ ]:
import sqlite3
import time

def sql(query):
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    conn.commit()
    if result is None:
        print('✅ Done')
        return
    if not result:
        print('(empty)')
        return
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')

def describe(table_name):
    print(f'\n📋 {table_name} 表结构：')
    print(f'   {"列名":15s} {"类型":10s} {"约束"}')
    print(f'   {"-"*15} {"-"*10} {"-"*20}')
    for row in conn.execute(f"PRAGMA table_info('{table_name}')"):
        constraints = []
        if row[3]: constraints.append('NOT NULL')
        if row[5]: constraints.append('PRIMARY KEY')
        constr = ', '.join(constraints) if constraints else ''
        print(f'   {row[1]:15s} {row[2]:10s} {constr}')
    fks = conn.execute(f"PRAGMA foreign_key_list('{table_name}')").fetchall()
    for fk in fks:
        print(f'   → FK: {fk[3]} → {fk[2]}({fk[4]})  ON DELETE {fk[6]}')

# 计时查询
def timed_query(query, label='Query'):
    start = time.perf_counter()
    result = conn.execute(query).fetchall()
    elapsed = (time.perf_counter() - start) * 1000  # 转毫秒
    print(f'⏱ {label}: {elapsed:.2f} ms  ({len(result)} rows)')
    return result


conn = sqlite3.connect('store.db')
conn.execute("PRAGMA foreign_keys = ON")

# ========== 建表 ==========
conn.executescript('''
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS accounts;
DROP TABLE IF EXISTS transfer_log;
DROP TABLE IF EXISTS orders;

-- 商品表（大量数据用于索引演示）
CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL CHECK (price > 0),
    stock INTEGER DEFAULT 0,
    rating REAL DEFAULT 0,
    is_active INTEGER DEFAULT 1
);

-- 账户表（用于事务演示）
CREATE TABLE accounts (
    id INTEGER PRIMARY KEY,
    owner TEXT NOT NULL,
    balance REAL NOT NULL DEFAULT 0 CHECK (balance >= 0),
    created_at TEXT DEFAULT (date('now'))
);

-- 转账记录
CREATE TABLE transfer_log (
    id INTEGER PRIMARY KEY,
    from_id INTEGER,
    to_id INTEGER,
    amount REAL NOT NULL,
    status TEXT DEFAULT 'pending',
    transferred_at TEXT DEFAULT (datetime('now')),
    FOREIGN KEY (from_id) REFERENCES accounts(id),
    FOREIGN KEY (to_id) REFERENCES accounts(id)
);

-- 订单表（用于部分索引演示）
CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    product_id INTEGER NOT NULL,
    quantity INTEGER DEFAULT 1,
    status TEXT DEFAULT 'pending',
    total REAL,
    created_at TEXT DEFAULT (date('now')),
    FOREIGN KEY (product_id) REFERENCES products(id)
);
''')

# ========== 填充数据 ==========
# 生成 2000 个商品
categories = ['Electronics', 'Books', 'Clothing', 'Food', 'Sports',
              'Home', 'Beauty', 'Toys', 'Music', 'Garden']
import random
random.seed(42)

products_data = []
for i in range(1, 2001):
    cat = categories[i % len(categories)]
    price = round(random.uniform(5, 500), 2)
    stock = random.randint(0, 200)
    rating = round(random.uniform(1, 5), 1)
    active = 1 if random.random() > 0.1 else 0  # 90% active
    products_data.append((f'Product_{i:04d}', cat, price, stock, rating, active))

conn.executemany(
    'INSERT INTO products (name, category, price, stock, rating, is_active) VALUES (?,?,?,?,?,?)',
    products_data
)

# 订单数据（部分已完成，部分 pending）
order_statuses = ['completed'] * 70 + ['pending'] * 20 + ['cancelled'] * 10
random.shuffle(order_statuses)
for i in range(1, 101):
    pid = random.randint(1, 2000)
    qty = random.randint(1, 5)
    total = round(qty * conn.execute('SELECT price FROM products WHERE id = ?', (pid,)).fetchone()[0], 2)
    conn.execute(
        'INSERT INTO orders (product_id, quantity, status, total) VALUES (?,?,?,?)',
        (pid, qty, order_statuses[i-1], total)
    )

# 账户
conn.executescript('''
INSERT INTO accounts (owner, balance) VALUES
    ('Alice',   5000),
    ('Bob',     3000),
    ('Charlie', 2000),
    ('Diana',   1000);
''')

conn.commit()

print('✅ store 数据库就绪！')
print(f'   products: {conn.execute("SELECT COUNT(*) FROM products").fetchone()[0]} 行')
print(f'   orders:   {conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]} 行')
print(f'   accounts: {conn.execute("SELECT COUNT(*) FROM accounts").fetchone()[0]} 行')
print(f'\n💡 2000 行商品数据，足够演示索引加速效果')
print(f'   10 个分类: {categories}')


In [ ]:
# 快速预览
print('📦 products（前 5 行）：')
sql('SELECT * FROM products LIMIT 5;')
print('\n💰 accounts：')
sql('SELECT * FROM accounts;')
print('\n📋 orders 状态分布：')
sql("SELECT status, COUNT(*) FROM orders GROUP BY status;")


---
## 6.1 索引（Index）

索引 = 数据库的「目录」。没有索引 → 全表扫描（逐行比对，O(n)）；有索引 → B-Tree 查找（O(log n)）。

```
没有索引：                              有索引：
SELECT ... WHERE category = 'Books'      SELECT ... WHERE category = 'Books'
  ↓                                       ↓
扫描 2000 行                               B-Tree 查 'Books' → 3 步定位
逐一比对 category                         直接跳到匹配行
  ↓                                       ↓
200 行结果                                200 行结果
耗时：~2 ms（小表看不出）                  耗时：~0.5 ms
```

In [ ]:
# 没有索引时：全表扫描
print('⚠️ 此时 products 表只有主键 id 的自动索引，category 列没有索引')
print()

# 多次查询取平均
print('== 无索引：按 category 查询 ==')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第1次')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第2次')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第3次')

print('\n== 无索引：按 rating 范围查询 ==')
timed_query("SELECT * FROM products WHERE rating > 4.5", '第1次')
timed_query("SELECT * FROM products WHERE rating > 4.5", '第2次')


In [ ]:
# EXPLAIN QUERY PLAN — 看数据库怎么执行
print('无索引时的查询计划：')
print()
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE category = 'Books';")
print('\n↑ SCAN TABLE products → 全表扫描，遍历所有 2000 行')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE rating > 4.5;")
print('↑ 同样是 SCAN TABLE')


In [ ]:
# 创建索引
print('在 category 列上建索引：')
sql("CREATE INDEX idx_products_category ON products(category);")

print('在 rating 列上建索引：')
sql("CREATE INDEX idx_products_rating ON products(rating);")

print('\n查看所有索引：')
sql("SELECT type, name, tbl_name FROM sqlite_master WHERE type = 'index';")


In [ ]:
# 有索引后：B-Tree 查找
print('== 有索引：按 category 查询 ==')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第1次')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第2次')
timed_query("SELECT * FROM products WHERE category = 'Books'", '第3次')

print('\n== 有索引：按 rating 范围查询 ==')
timed_query("SELECT * FROM products WHERE rating > 4.5", '第1次')
timed_query("SELECT * FROM products WHERE rating > 4.5", '第2次')

print('\n💡 对于 2000 行的小表，索引加速可能不明显（都在内存里）')
print('   但对于百万行级别的大表，索引是秒级 vs 毫秒级的差距')


In [ ]:
# 建索引后的查询计划
print('有索引后的查询计划：')
print()
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE category = 'Books';")
print('\n↑ SEARCH ... USING INDEX → 用索引查找，不再全表扫描！')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE rating > 4.5;")
print('↑ 同样用上了索引')


In [ ]:
# 复合索引：category + price 组合查询
sql("CREATE INDEX idx_products_cat_price ON products(category, price);")

print('复合索引 (category, price) 对以下查询有效：')
print()

print('✅ WHERE category = ? （前缀列）')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE category = 'Sports';")

print('\n✅ WHERE category = ? AND price > ? （全匹配 + 范围）')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE category = 'Sports' AND price > 100;")

print('\n❌ WHERE price > ?（跳过前缀 category，用不上复合索引）')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE price > 100;")
print('\n💡 复合索引遵循最左前缀原则：必须从最左列开始匹配')


In [ ]:
# 部分索引：只索引活跃商品（is_active = 1），省空间
sql("CREATE INDEX idx_active_products_price ON products(price) WHERE is_active = 1;")

print('部分索引：只索引 is_active=1 的商品')
print('\n查询活跃商品（命中部分索引）：')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE is_active = 1 AND price < 50;")

print('\n查询所有商品（部分索引用不上）：')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE price < 50;")

print('\n活跃 vs 不活跃商品数：')
sql("SELECT is_active, COUNT(*) FROM products GROUP BY is_active;")
print('如果 90% 都是活跃的，部分索引省不了太多；')
print('如果只有 10% 活跃（如订单 pending），省 90% 索引空间')


In [ ]:
# 索引的写入代价
print('创建一张临时表测试写入性能：')
conn.executescript('''
DROP TABLE IF EXISTS test_no_idx;
DROP TABLE IF EXISTS test_with_idx;

CREATE TABLE test_no_idx (id INTEGER PRIMARY KEY, val TEXT);
CREATE TABLE test_with_idx (id INTEGER PRIMARY KEY, val TEXT);
CREATE INDEX idx_val ON test_with_idx(val);
''')

# 测试 INSERT 性能
import time

data = [(f'value_{i}',) for i in range(1000)]

start = time.perf_counter()
conn.executemany('INSERT INTO test_no_idx (val) VALUES (?)', data)
no_idx_time = (time.perf_counter() - start) * 1000

start = time.perf_counter()
conn.executemany('INSERT INTO test_with_idx (val) VALUES (?)', data)
with_idx_time = (time.perf_counter() - start) * 1000

print(f'无索引的 INSERT 1000 行: {no_idx_time:.2f} ms')
print(f'有索引的 INSERT 1000 行: {with_idx_time:.2f} ms')
print(f'多花时间: {with_idx_time - no_idx_time:.2f} ms')
print('\n💡 索引加速查询，但拖慢写入——不是越多越好')

conn.executescript('DROP TABLE test_no_idx; DROP TABLE test_with_idx;')


### ✏️ 自己动手

给 `orders` 表的 `status` 列建一个索引，然后用 EXPLAIN QUERY PLAN 看查 pending 订单是否用上了索引。

In [ ]:
sql("""
-- CREATE INDEX idx_orders_status ON orders(status);
-- EXPLAIN QUERY PLAN SELECT * FROM orders WHERE status = 'pending';
""")


---
## 6.2 EXPLAIN QUERY PLAN 深入

`EXPLAIN QUERY PLAN` 是调优 SQL 的第一步——先看数据库怎么执行，再决定要不要加索引。

In [ ]:
# EXPLAIN QUERY PLAN 实战
print('① 简单 WHERE：')
sql("EXPLAIN QUERY PLAN SELECT * FROM products WHERE id = 42;")
print('↑ id 是 PRIMARY KEY，自动有索引 → SEARCH USING INTEGER PRIMARY KEY')

print('\n② ORDER BY + LIMIT：')
sql("EXPLAIN QUERY PLAN SELECT * FROM products ORDER BY price DESC LIMIT 10;")
print('↑ 没有 price 上的排序索引 → SCAN + 临时排序（USE TEMP B-TREE FOR ORDER BY）')

print('\n③ JOIN 查询计划：')
sql("EXPLAIN QUERY PLAN SELECT o.id, p.name FROM orders o JOIN products p ON o.product_id = p.id WHERE o.status = 'pending';")
print('↑ 先 SCAN orders，再用 product_id 的 FK 查找 products')


### ✏️ 自己动手

用 EXPLAIN QUERY PLAN 看看「查价格 < 20 且 rating > 4.0 的商品」的执行计划。然后决定要不要加索引。

In [ ]:
sql("""
-- EXPLAIN QUERY PLAN SELECT * FROM products WHERE price < 20 AND rating > 4.0;
""")


---
## 6.3 VACUUM

SQLite 删除数据后**不会自动释放磁盘空间**——VACUUM 重建数据库文件来回收空间。

In [ ]:
# VACUUM 演示：创建临时表，删数据，看概念
conn.executescript('''
DROP TABLE IF EXISTS vacuum_test;
CREATE TABLE vacuum_test (id INTEGER PRIMARY KEY, data TEXT);
''')

# 插入 5000 行
print('插入 5000 行数据...')
conn.executemany('INSERT INTO vacuum_test (data) VALUES (?)',
                 [(f'data_row_{i}' * 50,) for i in range(5000)])
conn.commit()

# 获取文件大小
import os
db_path = 'store.db'
size_before = os.path.getsize(db_path)
print(f'数据库文件大小（插入 5000 行后）: {size_before:,} bytes')

# 删除 80% 的数据
conn.execute('DELETE FROM vacuum_test WHERE id % 5 != 0;')  # 保留每 5 行中的 1 行
conn.commit()
size_after_delete = os.path.getsize(db_path)
print(f'数据库文件大小（删除 80% 后）: {size_after_delete:,} bytes')
print(f'⚠️ 删了 80% 数据，文件大小几乎不变！(差 {size_after_delete - size_before} bytes)')

# VACUUM
conn.execute('VACUUM;')
size_after_vacuum = os.path.getsize(db_path)
print(f'数据库文件大小（VACUUM 后）: {size_after_vacuum:,} bytes')
print(f'✅ VACUUM 回收了 {size_after_delete - size_after_vacuum:,} bytes')

conn.execute('DROP TABLE vacuum_test;')
conn.execute('VACUUM;')  # 再整理一次
print(f'\n清理后: {os.path.getsize(db_path):,} bytes')


### ✏️ 自己动手

VACUUM 命令除了回收空间还有什么副作用？（提示：想想 B-Tree 索引的碎片整理）

In [ ]:
# 思考：VACUUM 会重建整个数据库文件，所以它还能：
# 1. 整理索引碎片 → 查询性能提升
# 2. 压缩数据库文件 → 减少磁盘占用
# 3. 但会锁全库 → 执行期间不能读写
print('VACUUM = 数据库的"磁盘碎片整理"')


---
## 6.4 事务（Transactions）

事务 = 一组操作，要么全成功，要么全回滚。核心命令：`BEGIN` → 操作 → `COMMIT` / `ROLLBACK`。

最经典的例子：**银行转账**。A 扣 100，B 加 100——这两步必须同时成功或同时失败。

In [ ]:
# 转账前看一下余额
print('转账前：')
sql('SELECT id, owner, balance FROM accounts ORDER BY id;')

# 正确的事务转账：Alice → Bob 转 500
sql('''
BEGIN;
UPDATE accounts SET balance = balance - 500 WHERE id = 1;
UPDATE accounts SET balance = balance + 500 WHERE id = 2;
INSERT INTO transfer_log (from_id, to_id, amount, status) VALUES (1, 2, 500, 'completed');
COMMIT;
''')

print('\n转账后（Alice -500, Bob +500）：')
sql('SELECT id, owner, balance FROM accounts ORDER BY id;')
print('\n转账记录：')
sql('SELECT * FROM transfer_log;')

# 验证总额不变
total = conn.execute('SELECT SUM(balance) FROM accounts').fetchone()[0]
print(f'\n💰 总余额: {total} （转账前后不变 = 一致性 ✅）')


In [ ]:
# ROLLBACK：转账出错时回滚
print('转账前：')
sql('SELECT id, owner, balance FROM accounts ORDER BY id;')

# 模拟转账中出错
print('\n尝试 Bob → Charlie 转 10000（但 Charlie 的账户有 CHECK 约束...）')
print('注意：这里我们故意触发一个错误来演示 ROLLBACK')

# 手动操作 + try/except 模拟
try:
    conn.execute('BEGIN;')
    conn.execute('UPDATE accounts SET balance = balance - 10000 WHERE id = 2;')  # Bob 扣 10000
    # 这会让 Bob 余额变负 → CHECK 约束阻止
    conn.execute('COMMIT;')
except Exception as e:
    conn.execute('ROLLBACK;')
    print(f'❌ 事务回滚了！原因: {e}')

print('\n回滚后——余额没变：')
sql('SELECT id, owner, balance FROM accounts ORDER BY id;')
print('\n✅ 原子性：扣钱失败，加钱也不会执行，数据完好')


In [ ]:
# 对比：没有事务的"转账"有多危险
# 创建一个临时账户表模拟
conn.executescript('''
DROP TABLE IF EXISTS test_acc;
CREATE TABLE test_acc (id INTEGER PRIMARY KEY, name TEXT, balance REAL);
INSERT INTO test_acc VALUES (1, 'X', 1000), (2, 'Y', 500);
''')

print('没有事务保护的情况下分两步操作：')
print('初始状态：')
sql('SELECT * FROM test_acc;')

# 模拟：第一步成功，第二步因为某种原因没执行
conn.execute('UPDATE test_acc SET balance = balance - 200 WHERE id = 1;')
print('\nStep 1: X 扣 200 ✅')
sql('SELECT * FROM test_acc;')

# 假设这里程序崩溃了，第二步没执行
print('Step 2: （模拟崩溃——Y 没收到钱）❌')
print('\n最终状态：钱没了！')
print(f'总余额: {conn.execute("SELECT SUM(balance) FROM test_acc").fetchone()[0]} （应该 = 1500）')
print('\n💡 这就是为什么需要事务：防止"只做了一半"的操作')

conn.execute('DROP TABLE test_acc;')


### ✏️ 自己动手

写一个事务：Diana 转 300 给 Charlie，同时记录到 transfer_log。如果 Diana 余额不足（< 300），ROLLBACK。

In [ ]:
sql("""
-- 在这里写转账事务
""")


---
## 6.5 ACID 特性

```
A — Atomicity   原子性：事务不可分割，全做或全不做
C — Consistency 一致性：事务前后数据都满足约束
I — Isolation   隔离性：并发事务互不干扰
D — Durability  持久性：COMMIT 后数据永久保存
```

In [ ]:
# 演示 ACID 的四个特性

# A — 原子性（已在上面的 ROLLBACK 演示）
print('=== A: Atomicity 原子性 ===')
print('转账要么两步都成功，要么都回滚——不会出现扣了钱没加钱')

# C — 一致性
print('\n=== C: Consistency 一致性 ===')
total_before = conn.execute('SELECT SUM(balance) FROM accounts').fetchone()[0]
print(f'转账前总余额: {total_before}')

# 执行一个小额转账
conn.executescript('''
BEGIN;
UPDATE accounts SET balance = balance - 100 WHERE id = 1;
UPDATE accounts SET balance = balance + 100 WHERE id = 3;
COMMIT;
''')

total_after = conn.execute('SELECT SUM(balance) FROM accounts').fetchone()[0]
print(f'转账后总余额: {total_after}')
print(f'总额不变 ✅  (差 {total_after - total_before})')

# D — 持久性
print('\n=== D: Durability 持久性 ===')
print('COMMIT 后的数据写入磁盘（.db 文件），即使关闭 Python 重开也不丢')
# 验证
conn.execute('BEGIN;')
conn.execute("UPDATE accounts SET balance = balance - 1 WHERE id = 1;")
conn.execute('COMMIT;')
bal = conn.execute('SELECT balance FROM accounts WHERE id = 1').fetchone()[0]
print(f'Alice 余额: {bal}（已持久化到 store.db 文件）')

# I — 隔离性（概念演示）
print('\n=== I: Isolation 隔离性 ===')
print('SQLite 默认 SERIALIZABLE 级别：同一时间只有一个事务能写')
print('多个连接可以同时读（特别是开启 WAL 模式后）')

# 恢复
conn.execute('UPDATE accounts SET balance = balance + 1 WHERE id = 1;')
conn.commit()


In [ ]:
# WAL 模式 — 提升并发读性能
print('当前 journal 模式：')
sql('PRAGMA journal_mode;')

sql('PRAGMA journal_mode = WAL;')
print('\n切换后：')
sql('PRAGMA journal_mode;')
print('\nWAL (Write-Ahead Logging) 的好处：')
print('  ✅ 读操作不阻塞写操作')
print('  ✅ 写操作不阻塞读操作')
print('  ⚠️ 但同一时间仍然只有一个写者')
print('  ⚠️ 会产生额外的 -wal 和 -shm 文件')
print('\n对读多写少的 Web 应用，WAL 模式是标配')


In [ ]:
# 隔离性演示：两个"连接"同时操作
# 用同一个连接模拟（真实场景中应该是两个独立的 sqlite3.connect()）
import sqlite3 as sq3

print('模拟两个连接并发操作：')

# 连接 1：开始事务，修改但未提交
conn2 = sq3.connect('store.db')
conn2.execute('BEGIN;')
conn2.execute('UPDATE accounts SET balance = 9999 WHERE id = 4;')
print('连接 1: UPDATE Diana → 9999（未提交）')

# 连接 2（即当前 conn）：尝试读
bal = conn.execute('SELECT balance FROM accounts WHERE id = 4').fetchone()[0]
print(f'连接 2: 读取 Diana 余额 = {bal}')
print('  ↑ SQLite SERIALIZABLE 隔离 → 读不到未提交的修改')

# 连接 1 提交
conn2.execute('COMMIT;')
print('连接 1: COMMIT')

bal = conn.execute('SELECT balance FROM accounts WHERE id = 4').fetchone()[0]
print(f'连接 2: 再读 Diana 余额 = {bal}')
print('  ↑ COMMIT 后可见')

# 恢复
conn.execute('UPDATE accounts SET balance = 1000 WHERE id = 4;')
conn.commit()
conn2.close()


### ✏️ 自己动手

写一个事务：批量更新——给所有 rating > 4.5 的商品打 9 折（price * 0.9）。如果不小心把 price 设成负数，ROLLBACK 应该能阻止。

In [ ]:
sql("""
-- BEGIN;
-- UPDATE products SET price = price * 0.9 WHERE rating > 4.5;
-- 检查是否有 price <= 0 的...
-- COMMIT; 或 ROLLBACK;
""")


---
## 🎯 综合挑战

In [ ]:
# Q1：给 orders 表找出最合适的索引策略，用 EXPLAIN QUERY PLAN 验证
#     提示：orders 最常被按 status 查询、按 product_id JOIN products
sql("""
-- 分析以下两个查询：
-- EXPLAIN QUERY PLAN SELECT * FROM orders WHERE status = 'pending';
-- EXPLAIN QUERY PLAN SELECT * FROM orders o JOIN products p ON o.product_id = p.id;
-- 然后决定建什么索引
""")


In [ ]:
# Q2：用事务实现一个"下单"流程：
#     ① 检查库存 stock > 0
#     ② 扣库存 UPDATE products SET stock = stock - 1
#     ③ 创订单 INSERT INTO orders
#     ④ 如果任何一步失败，ROLLBACK
sql("""
-- 在这里写下单事务
""")


In [ ]:
# Q3：查 products 表中 rating 最高的 5 个商品的执行计划
#     建索引前后对比
sql("""
-- EXPLAIN QUERY PLAN SELECT * FROM products ORDER BY rating DESC LIMIT 5;
-- CREATE INDEX idx_... ON products(...);
-- EXPLAIN QUERY PLAN SELECT * FROM products ORDER BY rating DESC LIMIT 5;
""")


In [ ]:
# Q4：想一个用部分索引的场景，写出来
#     提示：orders 表的 status 列，只有 'pending' 状态的订单经常被查
sql("""
-- CREATE INDEX idx_pending_orders ON orders(...) WHERE status = 'pending';
""")


In [ ]:
# Q5：写一段代码，用 Python 的 try/except + sqlite3 实现事务的自动回滚
#     提示：conn.execute('BEGIN') ... try ... conn.execute('COMMIT') ... except ... conn.execute('ROLLBACK')
sql("""
-- 在这个 cell 里用 Python 的 try/except 实现（而不是纯 SQL）
""")
print('提示：用 conn.execute("BEGIN") / conn.execute("COMMIT") / conn.execute("ROLLBACK")')


---
## ✅ 检查清单

- [ ] 理解索引的 B-Tree 原理，知道为什么 O(log n) 比 O(n) 快
- [ ] 能用 CREATE INDEX 创建单列索引、复合索引、部分索引
- [ ] 理解复合索引的最左前缀原则
- [ ] 能用 EXPLAIN QUERY PLAN 分析查询是否使用了索引
- [ ] 能区分 SCAN TABLE 和 SEARCH ... USING INDEX
- [ ] 知道索引的代价：拖慢写入、占磁盘空间
- [ ] 理解 VACUUM 的作用和适用场景
- [ ] 能用 BEGIN / COMMIT / ROLLBACK 写事务
- [ ] 理解 ACID 四个字母分别代表什么
- [ ] 能解释为什么转账必须用事务
- [ ] 理解隔离级别和并发问题（脏读、不可重复读、幻读）
- [ ] 知道 SQLite 的 WAL 模式及其好处

---

> 📖 下一章：[07 - 数据库系统与安全](../07-database-systems-and-security/) — MySQL/PostgreSQL 对比、SQL 注入防御